# 03 · Join Sofascore + Capology — France Ligue 1 25/26 (snapshot 20260428)

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2025/26 de la Ligue 1 francesa**.

⚠️ **Nota sobre el snapshot:** la temporada 25/26 está aún en curso. Se trabaja con
una foto fija de Sofascore (`df_france_2526_snapshot_20260428.csv`). Este notebook
deberá reejecutarse con los datos definitivos cuando finalice la liga, generando
entonces el master sin sufijo de fecha (`master_france_2526.csv`).

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_france_2526_snapshot_20260428.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_france_2526.csv').copy()

print(f'Sofascore (snapshot):  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:              {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore (snapshot):  537 jugadores | 117 columnas
Capology:              501 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   as monaco
   olympique de marseille
   olympique lyonnais
   paris saint germain
   rc lens
   rc strasbourg
   stade brestois
   stade rennais

En Capology pero no en Sofascore:
   brest
   lens
   lyon
   marseille
   monaco
   psg
   rennes
   strasbourg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [9]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brest':'stade brestois',
            'lens':'rc lens',
            'lyon':'olympique lyonnais',
            'olympique marseille':'olympique de marseille',
            'monaco':'as monaco',
            'psg':'paris saint germain',
            'rennes':'stade rennais',
            'strasbourg':'rc strasbourg'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [10]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 412/537 (76.7%)
Sin emparejar: 125


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [11]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          9
Revisión media    (0.75 ≤ score < 0.90):   13
Revisión estricta (0.50 ≤ score < 0.75):   55
Revisión muy est. (score < 0.50):           48


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [12]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
51,Radosław Majecki,Stade Brestois,radoslaw majecki,0.968
9,Emmanuel Emegha,RC Strasbourg,emanuel emegha,0.966
52,Waren Kamanzi,Toulouse,warren kamanzi,0.963
80,Darlin Yongwa,Lorient,darline yongwa,0.963
45,Luc Zogbé,Stade Brestois,luck zogbe,0.947
32,Habibou Diallo,Metz,habib diallo,0.923
96,Elias Legendre Quiñonez Zae,Stade Rennais,elias legendre quinonez,0.920
16,Ali Youssef,Nantes,ali youssif,0.909
54,Bahereba Guirassy,Nantes,herba guirassy,0.903


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [13]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
13,Mousa Tamari,Stade Rennais,mousa al tamari,0.889
35,Ibou Sane,Metz,sadibou sane,0.857
4,Matías Fernández,Lille,matias fernandez pardo,0.842
28,Urie-Michel Mboula,Metz,michel mboula,0.839
27,Ignatius Kpene Ganago,Nantes,ignatius ganago,0.833
53,Morgan Bokele Mputu,Metz,morgan bokele,0.812
34,Tim Weah,Olympique de Marseille,timothy weah,0.800
5,Arthur Avom Ebong,Lorient,arthur avom,0.786
33,Stanis Idumbo Muzambo,AS Monaco,stanis idumbo,0.765
94,Samir Sophian Chergui,Paris FC,samir chergui,0.765


In [14]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['ibou sane'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 12 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [15]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
56,Daren Nbenbege Mosengo,Le Havre,daren mosengo,0.743
39,Amir Murillo,Olympique de Marseille,michael murillo,0.741
18,Ahmadou Bamba Dieng,Lorient,bamba dieng,0.733
49,Conrad Jaden Egan-Riley,Olympique de Marseille,cj egan riley,0.722
47,Alexsandro Ribeiro,Lille,alexsandro,0.714
74,Joel Mugisha Mvuka,Lorient,joel mvuka,0.714
73,Cheick Tidiane Sabaly,Metz,cheikh sabaly,0.706
93,Ally Samatta,Le Havre,mbwana samatta,0.692
26,Abdulay Juma Bah,Nice,juma bah,0.667
12,Moustapha Mbow,Paris FC,mamadou mbow,0.615


In [19]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['daren nbenbege mosengo',
                    'amir murillo',
                    'ahmadou bamba dieng',
                    'conrad jaden egan riley',
                    'alexsandro ribeiro',
                    'joel mugisha mvuka',
                    'cheick tidiane sabaly',
                    'ally samatta',
                    'abdulay juma bah',
                    'moustapha mbow',
                    'serigne diop',
                    'danny namaso',
                    'joseph n duquidi',
                    'michel diaz',
                    'karim dermane',
                    'abner vinicius',
                    'djibirin harouna'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 17


### 7.4 Revisión muy estricta (score < 0.50)

Candidatos con muy baja similitud. Por defecto ninguno se acepta.
Añadir a `ACCEPT_VERY_LOW_FUZZY` los que se confirmen manualmente.

In [20]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
6,Brad-Hamilton Mantsounga,Nice,morgan sanson,0.486
7,Julien Le Cardinal,Stade Brestois,romain del castillo,0.486
3,Georges Mikautadze,Olympique Lyonnais,moussa niakhate,0.485
114,Justin Bourgault,Stade Brestois,lucas tousart,0.483
67,Younes Namli,Le Havre,lionel mpasi nzau,0.483
29,Robinio Vaz,Olympique de Marseille,geronimo rulli,0.480
50,Gabriel Osho,Auxerre,gideon mensah,0.480
106,Enzo Molebe,Olympique Lyonnais,afonso moreira,0.480
59,Ilyas Azizi,Toulouse,niklas schmidt,0.480
21,Pablo Rosario,Nice,ali abdi,0.476


In [21]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [22]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 450/537 (83.8%)
Sin salario:     87


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [23]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 87


,player,team,minutesPlayed,appearances,goals,assists
0,George Ilenikhena,AS Monaco,386,15,2,0
1,Pape Cabral,AS Monaco,73,4,0,0
2,Eliesse Ben Seghir,AS Monaco,43,2,0,0
3,Samuel Nibombe,AS Monaco,1,1,0,0
4,Sidiki Cherif,Angers,1164,19,4,0
5,Justin-Noel Kalumba,Angers,225,4,0,0
6,Bané Diatta,Angers,9,1,0,0
7,Gabriel Osho,Auxerre,134,2,0,0
8,Theo Bair,Auxerre,23,2,0,0
9,Yvan Zaddy,Auxerre,1,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [24]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  AS Monaco  —  SF sin salario:


,player,minutesPlayed
0,Eliesse Ben Seghir,43
1,George Ilenikhena,386
2,Pape Cabral,73
3,Samuel Nibombe,1


  CG plantilla completa:


,player,player_norm
0,Aladji Bamba,aladji bamba
1,Aleksandr Golovin,aleksandr golovin
2,Ansu Fati,ansu fati
3,Caio Henrique,caio henrique
4,Christian Mawissa,christian mawissa
5,Denis Zakaria,denis zakaria
6,Edan Diop,edan diop
7,Eric Dier,eric dier
8,Folarin Balogun,folarin balogun
9,Jordan Teze,jordan teze



  Angers  —  SF sin salario:


...

,player,player_norm
0,Abdelhamid Ait Boudlal,abdelhamid ait boudlal
1,Alidu Seidu,alidu seidu
2,Anthony Rouault,anthony rouault
3,Arnaud Nordin,arnaud nordin
4,Breel Embolo,breel embolo
5,Brice Samba,brice samba
6,Djaoui Cissé,djaoui cisse
7,Elías Legendre Quiñónez,elias legendre quinonez
8,Estéban Lepaul,esteban lepaul
9,Glen Kamara,glen kamara



  Toulouse  —  SF sin salario:


,player,minutesPlayed
0,Ilyas Azizi,22
1,Jaydee Canvot,180
2,Mathis Saka,8


  CG plantilla completa:


,player,player_norm
0,Abu Francis,abu francis
1,Álex Domínguez,alex dominguez
2,Alexis Vossah,alexis vossah
3,Aron Dønnum,aron dnnum
4,Charlie Cresswell,charlie cresswell
5,Cristian Cásseres Jr,cristian casseres jr
6,Dayann Methalie,dayann methalie
7,Djibril Sidibé,djibril sidibe
8,Emersonn,emersonn
9,Enzo Faty,enzo faty


In [25]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')

Matches manuales definidos: 0


In [26]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 450/537 (83.8%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

⚠️ Nombre con sufijo `_snapshot_20260428` para diferenciar del master definitivo
que se generará al cierre de la temporada (`master_france_2526.csv`).

In [27]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_france_2526_snapshot_20260428.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_france_2526_snapshot_20260428.csv
   Jugadores totales:  537
   Con salario:        450
   Sin salario (NaN):  87
   Columnas:           122
